# Integral Form Bayesian Operator Inference: Compressible Euler

**Model E** — GP for smooth states + integral form ODE constraint.
No derivative estimation needed. The integral form is noise-robust
(integration smooths noise vs. differentiation which amplifies it)
and serves as an anchor preventing the null basin.

In [1]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import jax
import jax.numpy as jnp
import opinf
import matplotlib.pyplot as plt
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, Predictive, autoguide
from jax import random

from core import generate_trajectory, JaxCompatibleModel
sys.path.insert(0, '../euler')
import config
from config import Basis

numpyro.set_platform('cpu')
numpyro.set_host_device_count(4)
np.random.seed(42)
rng_key = random.PRNGKey(42)

In [2]:
# === EXPERIMENT CONFIGURATION ===
OPERATORS = "cAH"
IVP_METHOD = "RK45"
NUM_MODES = 6

# Data generation
TRAINING_SPAN = (0, 0.08)
PREDICTION_SPAN = (0, config.time_domain[-1])
NUM_SAMPLES = 250
NOISE_LEVEL = 0.03

# Model E hyperparameters
GAMMA = 10.0           # operator prior scale
GAMMA2 = 1.0           # integral constraint scale
WINDOW_SIZE = 10       # points per integration window

# GP priors (tight, informed)
GP_LENGTHSCALE_PRIOR_SCALE = 0.5
GP_NOISE_PRIOR_SCALE = 0.5

# SVI
GUIDE = autoguide.AutoNormal
NUM_SVI_STEPS = 20000
LEARNING_RATE = 1e-3
NUM_POSTERIOR_SAMPLES = 500

In [3]:
# Generate training data (same pattern as notebook 03)
(fom, time_domain_full, true_states, time_sampled, snapshots_sampled) = \
    generate_trajectory(config, config.time_domain, TRAINING_SPAN, NUM_SAMPLES, NOISE_LEVEL)

print(f"Full time domain: {time_domain_full.shape}")
print(f"True states: {true_states.shape}")
print(f"Sampled times: {time_sampled.shape}")
print(f"Training span: [{TRAINING_SPAN[0]:.4f}, {TRAINING_SPAN[1]:.4f}]")
print(f"Prediction span: [{PREDICTION_SPAN[0]:.4f}, {PREDICTION_SPAN[1]:.4f}]")

# Fit POD basis and compress
basis = Basis(num_vectors=NUM_MODES)
basis.fit(snapshots_sampled)

snapshots_comp_sampled = basis.compress(snapshots_sampled)
full_states_compressed = basis.compress(true_states)

print(f"Compressed shape: {snapshots_comp_sampled.shape}")
print(f"Cumulative energy: {basis.cumulative_energy:.4%}")

generating training data...done in 1.05 s.
Full time domain: (401,)
True states: (600, 401)
Sampled times: (250,)
Training span: [0.0000, 0.0800]
Prediction span: [0.0000, 0.1500]
Compressed shape: (6, 250)
Cumulative energy: 88.2756%


In [4]:
# Build ROM object for data matrix assembly (operator structure)
rom = opinf.ROM(
    basis=basis,
    ddt_estimator=opinf.ddt.NonuniformFiniteDifferencer(time_sampled),
    model=JaxCompatibleModel(
        operators=OPERATORS,
        solver=opinf.lstsq.L2Solver(regularizer=1e0),
    ),
)
rom.fit(states=snapshots_sampled)

r = NUM_MODES
d = rom.model.operator_matrix.shape[1]
print(f"Operator matrix shape: ({r}, {d})")

# Verify data matrix assembly shapes
x_test = jnp.array(snapshots_comp_sampled[:, 0:1])  # (r, 1)
f_test = rom.model._assemble_data_matrix(x_test, inputs=None)
print(f"f(x) shape for single state (r,1) input: {f_test.shape}  [expect (1, {d})]")

Operator matrix shape: (6, 28)
f(x) shape for single state (r,1) input: (1, 28)  [expect (1, 28)]


In [5]:
def build_integral_form_model(
    rom,
    num_modes,
    t_train_np,
    y_obs_np,
    noise_level,
    window_size=10,
    gp_lengthscale_prior_scale=0.5,
    gp_noise_prior_scale=0.5,
):
    """Build Model E: GP latent states + integral form ODE constraint.

    Returns a NumPyro model closure with precomputed kernel matrices.
    """
    t_train = jnp.array(t_train_np)
    n_train = len(t_train)
    T = float(t_train[-1] - t_train[0])
    y_obs = jnp.array(y_obs_np)

    # Precompute kernel distance matrix and identity
    sq_diffs_tt = (t_train[:, None] - t_train[None, :]) ** 2
    I_train = jnp.eye(n_train)

    # Precompute time steps for quadrature
    dt_arr = jnp.diff(t_train)

    # Precompute window indices
    n_windows = n_train // window_size
    window_starts = jnp.array([w * window_size for w in range(n_windows)])
    window_ends = jnp.array([
        min((w + 1) * window_size, n_train - 1) for w in range(n_windows)
    ])

    # Operator shape from ROM
    op_shape = rom.model.operator_matrix.shape  # (num_modes, d)

    # GP prior locations (tight, informed)
    ls_prior_loc = jnp.full(num_modes, jnp.log(T / 20.0))
    var_prior_loc = jnp.array([
        jnp.log(jnp.var(y_obs[i]) + 1e-8) for i in range(num_modes)
    ])
    noise_prior_loc = jnp.full(num_modes, jnp.log(noise_level))

    def _rbf_kernel(ell, sig2, sq_diffs, jitter):
        """RBF kernel from precomputed squared differences."""
        K = sig2 * jnp.exp(-0.5 * sq_diffs / ell ** 2)
        return K + jitter * jnp.eye(sq_diffs.shape[0])

    def _single_gp_forward(ell, sig2, X_raw, jitter):
        """Kernel -> Cholesky -> non-centered X for one mode."""
        eff_jitter = jnp.maximum(jitter, sig2 * 1e-4)
        K = _rbf_kernel(ell, sig2, sq_diffs_tt, eff_jitter)
        L = jnp.linalg.cholesky(K)
        X = L @ X_raw
        return X

    _batch_gp_forward = jax.vmap(_single_gp_forward, in_axes=(0, 0, 0, None))

    def model(gamma=1.0, gamma2=1.0, jitter=1e-4):
        # --- Sample GP hyperparameters per mode ---
        ells = jnp.stack([
            numpyro.sample(
                f"lengthscale_{i}",
                dist.LogNormal(ls_prior_loc[i], gp_lengthscale_prior_scale),
            )
            for i in range(num_modes)
        ])
        sig2s = jnp.stack([
            numpyro.sample(
                f"variance_{i}",
                dist.LogNormal(var_prior_loc[i], 0.5),
            )
            for i in range(num_modes)
        ])
        nus = jnp.stack([
            numpyro.sample(
                f"noise_{i}",
                dist.LogNormal(noise_prior_loc[i], gp_noise_prior_scale),
            )
            for i in range(num_modes)
        ])

        # --- Sample X_raw per mode (non-centered GP parameterisation) ---
        X_raws = jnp.stack([
            numpyro.sample(
                f"X_raw_{i}",
                dist.Normal(jnp.zeros(n_train), jnp.ones(n_train)),
            )
            for i in range(num_modes)
        ])

        # --- Batched GP forward: kernel, Cholesky, X = L @ X_raw ---
        Xs = _batch_gp_forward(ells, sig2s, X_raws, jitter)  # (num_modes, n_train)

        # Register latent states and observation likelihoods
        for i in range(num_modes):
            numpyro.deterministic(f"X_{i}", Xs[i])
            numpyro.sample(
                f"obs_{i}",
                dist.Normal(Xs[i], jnp.sqrt(nus[i])),
                obs=y_obs[i],
            )

        # --- Sample operator O ~ N(0, gamma * I) ---
        O = numpyro.sample(
            "O",
            dist.Normal(jnp.zeros(op_shape), gamma * jnp.ones(op_shape)),
        )

        # --- Compute operator dynamics at all training points ---
        # _assemble_data_matrix expects (r, n_points), returns (n_points, d)
        f_X_all = rom.model._assemble_data_matrix(Xs, inputs=None)  # (n_train, d)
        dynamics_all = f_X_all @ O.T  # (n_train, num_modes)

        # --- Integral form constraints over non-overlapping windows ---
        for w in range(n_windows):
            i_start = int(window_starts[w])
            i_end = int(window_ends[w])

            # State difference from GP latent states
            delta_X = Xs[:, i_end] - Xs[:, i_start]  # (num_modes,)

            # Trapezoidal quadrature: integral of dynamics over window
            # ∫ f(X(s)) O^T ds ≈ Σ_k dt_k * dynamics[k]
            dt_window = dt_arr[i_start:i_end]  # (window_size-1,) or less
            dyn_window = dynamics_all[i_start:i_end]  # (window_size-1, num_modes) or less
            integral = jnp.sum(
                dt_window[:, None] * dyn_window, axis=0
            )  # (num_modes,)

            # Likelihood: delta_X ~ N(integral, gamma2)
            numpyro.sample(
                f"integral_{w}",
                dist.Normal(integral, gamma2),
                obs=delta_X,
            )

    return model


model = build_integral_form_model(
    rom=rom,
    num_modes=NUM_MODES,
    t_train_np=time_sampled,
    y_obs_np=snapshots_comp_sampled,
    noise_level=NOISE_LEVEL,
    window_size=WINDOW_SIZE,
    gp_lengthscale_prior_scale=GP_LENGTHSCALE_PRIOR_SCALE,
    gp_noise_prior_scale=GP_NOISE_PRIOR_SCALE,
)

print(f"Model built with {NUM_MODES} modes, {len(time_sampled)} training points")
print(f"  Window size: {WINDOW_SIZE} -> {len(time_sampled) // WINDOW_SIZE} integral constraints per mode")
print(f"  Operator shape: {rom.model.operator_matrix.shape}")

Model built with 6 modes, 250 training points
  Window size: 10 -> 25 integral constraints per mode
  Operator shape: (6, 28)


In [6]:
guide = GUIDE(model)
optimizer = numpyro.optim.Adam(step_size=LEARNING_RATE)
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

svi_result = svi.run(
    rng_key,
    NUM_SVI_STEPS,
    gamma=GAMMA,
    gamma2=GAMMA2,
    progress_bar=True,
)

# Plot ELBO convergence
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(svi_result.losses)
ax.set_xlabel("Iteration")
ax.set_ylabel("ELBO Loss")
ax.set_title("Model E: SVI Convergence (Integral Form)")
plt.tight_layout()
plt.show()

print(f"Final ELBO loss: {svi_result.losses[-1]:.4f}")

  0%|          | 0/20000 [00:00<?, ?it/s]


ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape int32[]
The problem arose with the `int` function. If trying to convert the data type of a value, try using `x.astype(int)` or `jnp.array(x, int)` instead.
The error occurred while tracing the function body_fn at /Users/anthonypoole/miniconda3/envs/prob_rom/lib/python3.13/site-packages/numpyro/infer/svi.py:383 for jit. This value became a tracer due to JAX operations on these lines:

  operation a[35m:f32[250][39m = broadcast_in_dim[
  broadcast_dimensions=()
  shape=(250,)
  sharding=None
] 0.10000000149011612:f32[]
    from line /var/folders/z5/hb7nxgp524b67trvdyv41twh0000gn/T/ipykernel_91549/633230211.py:5:13 (<module>)

  operation a[35m:f32[250][39m = broadcast_in_dim[
  broadcast_dimensions=()
  shape=(250,)
  sharding=None
] 0.10000000149011612:f32[]
    from line /var/folders/z5/hb7nxgp524b67trvdyv41twh0000gn/T/ipykernel_91549/633230211.py:5:13 (<module>)

  

In [ ]:
params = svi_result.params
rng_key, sample_key, pred_key = random.split(rng_key, 3)

# Sample latent variables from the learned guide
model_kwargs = dict(gamma=GAMMA, gamma2=GAMMA2)
posterior_samples = guide.sample_posterior(
    sample_key, params, sample_shape=(NUM_POSTERIOR_SAMPLES,), **model_kwargs,
)

# Run predictive to get deterministic sites
predictive = Predictive(
    model,
    posterior_samples=posterior_samples,
    num_samples=NUM_POSTERIOR_SAMPLES,
)
model_output = predictive(pred_key, **model_kwargs)

# Merge guide samples + model output
samples = {**model_output, **posterior_samples}

# Operator summary statistics
O_samples = samples['O']
print(f"Operator samples shape: {O_samples.shape}")
print(f"Operator norm (median): {np.median(np.linalg.norm(O_samples, axis=(1, 2))):.4f}")

# GP hyperparameters
for i in range(NUM_MODES):
    ls = np.median(np.array(samples[f'lengthscale_{i}']))
    var = np.median(np.array(samples[f'variance_{i}']))
    noise = np.median(np.array(samples[f'noise_{i}']))
    print(f"  Mode {i}: ℓ={ls:.6f}, σ²={var:.6f}, ν={noise:.6f}")

In [ ]:
fig, axes = plt.subplots(2, NUM_MODES, figsize=(4 * NUM_MODES, 7), sharex=True)
if NUM_MODES == 1:
    axes = axes.reshape(2, 1)

for i in range(NUM_MODES):
    # Top row: GP latent state fit
    ax = axes[0, i]
    X_i_samples = np.array(samples[f'X_{i}'])  # (num_samples, n_train)
    median = np.median(X_i_samples, axis=0)
    q05 = np.percentile(X_i_samples, 5, axis=0)
    q95 = np.percentile(X_i_samples, 95, axis=0)

    ax.plot(time_sampled, snapshots_comp_sampled[i], 'k.', ms=3, alpha=0.4, label='Data')
    ax.plot(time_sampled, full_states_compressed[i, :len(time_sampled)],
            'g-', lw=1, alpha=0.5, label='Truth (train)')
    ax.plot(time_sampled, median, 'b-', lw=2, label='GP median')
    ax.fill_between(time_sampled, q05, q95, alpha=0.2, color='blue', label='5-95%')
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.set_ylabel('State')
        ax.legend(fontsize=7)

    # Bottom row: residuals (data - GP median)
    ax2 = axes[1, i]
    residuals = snapshots_comp_sampled[i] - median
    ax2.plot(time_sampled, residuals, 'r.', ms=2, alpha=0.5)
    ax2.axhline(0, color='gray', ls='--', alpha=0.5)
    ax2.set_title(f'Residuals mode {i}')
    if i == 0:
        ax2.set_ylabel('Residual')
    ax2.set_xlabel('Time')

fig.suptitle('Model E: GP Latent State Fit (Integral Form)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate ROM predictions using posterior operator samples
NUM_REGRESSION_POINTS = 150
time_domain_eval = np.linspace(PREDICTION_SPAN[0], PREDICTION_SPAN[1], NUM_REGRESSION_POINTS)

stable_count = 0
rom_solves = []

for s in range(min(200, NUM_POSTERIOR_SAMPLES)):
    O_s = np.array(O_samples[s])
    rom.model._extract_operators(O_s)
    try:
        sol = rom.model.predict(
            state0=np.array(snapshots_comp_sampled[:, 0]),
            t=time_domain_eval,
            method=IVP_METHOD,
        )
        if rom.model.predict_result_.y.shape[1] >= len(time_domain_eval):
            rom_solves.append(rom.model.predict_result_.y)
            stable_count += 1
    except Exception:
        pass

print(f"Stable ROM solves: {stable_count}/{min(200, NUM_POSTERIOR_SAMPLES)}")

# Plot ROM predictions vs truth
fig, axes = plt.subplots(1, NUM_MODES, figsize=(4 * NUM_MODES, 3.5), sharex=True)
if NUM_MODES == 1:
    axes = [axes]

for i in range(NUM_MODES):
    ax = axes[i]
    ax.plot(time_domain_full, full_states_compressed[i], 'k-', lw=1.5, label='Truth')
    ax.plot(time_sampled, snapshots_comp_sampled[i], 'k.', ms=3, alpha=0.4, label='Data')

    if rom_solves:
        rom_arr = np.array(rom_solves)
        med = np.median(rom_arr[:, i, :], axis=0)
        q05 = np.percentile(rom_arr[:, i, :], 5, axis=0)
        q95 = np.percentile(rom_arr[:, i, :], 95, axis=0)
        ax.plot(time_domain_eval, med, 'purple', ls='--', lw=2, label='ROM median')
        ax.fill_between(time_domain_eval, q05, q95, alpha=0.2, color='purple', label='5-95%')

    ax.axvline(TRAINING_SPAN[1], color='gray', ls=':', alpha=0.5)
    ax.set_title(f'Mode {i}')
    if i == 0:
        ax.legend(fontsize=7)
    ax.set_xlabel('Time')

fig.suptitle('Model E: ROM Predictions vs Truth (Integral Form)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === Experiment Summary ===
print('=' * 60)
print('Model E — Integral Form Bayesian Operator Inference')
print('=' * 60)
print(f'PDE: Compressible Euler equations')
print(f'Operator structure: {OPERATORS}')
print(f'POD modes: {NUM_MODES}')
print(f'Training span: {TRAINING_SPAN}')
print(f'Num observations: {NUM_SAMPLES}')
print(f'Noise level: {NOISE_LEVEL:.0%}')
print(f'Operator prior scale (gamma): {GAMMA}')
print(f'Integral constraint scale (gamma2): {GAMMA2}')
print(f'Window size: {WINDOW_SIZE}')
print(f'SVI steps: {NUM_SVI_STEPS}')
print(f'Posterior samples: {NUM_POSTERIOR_SAMPLES}')
print(f'Stable ROM solves: {stable_count}/{min(200, NUM_POSTERIOR_SAMPLES)}')
if rom_solves:
    O_med = np.median(np.array(O_samples), axis=0)
    print(f'Median operator Frobenius norm: {np.linalg.norm(O_med):.4f}')
print('=' * 60)